In [1]:
# ============================================================
# Stanford RNA 3D Folding — Tri-Model Hybrid Submission
#
# Combines three prediction sources per target:
#   1. Best Enhanced TBM  (template-based modelling, enhanced)
#   2. Best Boltz-2       (deep-learning, Boltz repeat-0)
#   3. Best RNAPro        (deep-learning, 500M model)
#   4. 50/50 blend        (Boltz + TBM, centroid-aligned)
#   5. 50/50 blend        (RNAPro + Boltz, centroid-aligned)
#
# Fallback tiers (applied per-target):
#   Tier A — all three available        → strategy above
#   Tier B — RNAPro unavailable         → TBM, Boltz, blend, hinge, jitter
#   Tier C — Boltz unavailable          → TBM, RNAPro, blend, hinge, jitter
#   Tier D — only TBM available         → 5 geometric variants of TBM
#
# All predictions are refined with adaptive_rna_constraints for
# physical validity (bond lengths / clash prevention).
# ============================================================

import os, sys, gc, json, shutil, time, warnings, random
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# SECTION 0 — Install dependencies & model sources
# ─────────────────────────────────────────────────────────────

IS_SCORING_RUN = os.environ.get('KAGGLE_IS_COMPETITION_RERUN')
print(f"IS_SCORING_RUN = {IS_SCORING_RUN}")

# ── Biopython ────────────────────────────────────────────────
# (Adjust wheel path for your Kaggle dataset input)
os.system(
    "pip install --no-index "
    "/kaggle/input/datasets/kami1976/biopython-cp312/"
    "biopython-1.86-cp312-cp312-manylinux2014_x86_64"
    ".manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl"
)

# ── Boltz ────────────────────────────────────────────────────
os.system("cp -r /kaggle/input/datasets/lbugnon/boltz-src-minimal ./")
os.system("pip install --no-index --no-build-isolation -e ./boltz-src-minimal")
os.system("mkdir -p boltz_cache")
os.system("cp -r /kaggle/input/datasets/lbugnon/boltz2 boltz_cache")
os.system("mv boltz_cache/boltz2/mols/mols/* boltz_cache/boltz2/mols/")
os.system("rm -r boltz_cache/boltz2/mols/mols/")
os.system("tar -cf boltz_cache/boltz2/mols.tar boltz_cache/boltz2/mols")

# ── RNAPro ───────────────────────────────────────────────────
os.system("cp -r /kaggle/input/rnapro-src/RNAPro .")
os.system("cp /kaggle/input/rnapro-src/rnapro-private-best-500m.ckpt .")
os.chdir("RNAPro")
os.system("pip install -e . --no-deps")
os.chdir("..")

# ─────────────────────────────────────────────────────────────
# SECTION 1 — Imports (all heavy imports after installs)
# ─────────────────────────────────────────────────────────────

from tqdm.auto import tqdm
from collections import Counter
from random import shuffle
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.spatial import distance_matrix
from scipy.spatial.transform import Rotation as R_scipy

from Bio.Align import PairwiseAligner
from Bio.Seq import Seq

seed = 21
np.random.seed(seed)
random.seed(seed)

# ─────────────────────────────────────────────────────────────
# SECTION 2 — Data ingestion
# ─────────────────────────────────────────────────────────────

BASE_PATH = '/kaggle/input/stanford-rna-3d-folding-2'

test_seqs        = pd.read_csv(f'{BASE_PATH}/test_sequences.csv')
train_seqs       = pd.read_csv(f'{BASE_PATH}/train_sequences.csv')
validation_seqs  = pd.read_csv(f'{BASE_PATH}/validation_sequences.csv')
train_labels     = pd.read_csv(f'{BASE_PATH}/train_labels.csv', low_memory=False)
validation_labels = pd.read_csv(f'{BASE_PATH}/validation_labels.csv')
sample_submission = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')

print(f"✓ {len(train_seqs)} training | {len(validation_seqs)} validation | {len(test_seqs)} test sequences")

# ── FASTA / stoichiometry helpers ────────────────────────────

nucleotides = {"A", "G", "C", "U"}
aminoacids  = {"A","R","N","D","C","E","Q","G","H","I","L","K","M","F","P","S","T","W","Y","V"}

def parse_fasta(fasta_content: str):
    out, cur, seq_parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(seq_parts)
            cur = line[1:].split()[0]; seq_parts = []
        else:
            seq_parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(seq_parts)
    return out

def parse_stoichiometry(stoich: str):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    return [(p.split(":")[0].strip(), int(p.split(":")[1]))
            for p in str(stoich).split(";")]

def get_chain_segments(row):
    seq, stoich, all_seq = row["sequence"], row.get("stoichiometry", ""), row.get("all_sequences", "")
    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip() == "" or str(all_seq).strip() == "":
        return [(0, len(seq))]
    try:
        chain_dict = parse_fasta(all_seq)
        order      = parse_stoichiometry(stoich)
        segs, pos  = [], 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                segs.append((pos, pos + len(base))); pos += len(base)
        return segs if pos == len(seq) else [(0, len(seq))]
    except Exception:
        return [(0, len(seq))]

def build_segments_map(df):
    return {r["target_id"]: get_chain_segments(r) for _, r in df.iterrows()}

test_segs_map  = build_segments_map(test_seqs)
train_segs_map = build_segments_map(train_seqs)

# ─────────────────────────────────────────────────────────────
# SECTION 3 — Labels → coordinate dictionaries
# ─────────────────────────────────────────────────────────────

def process_labels(labels_df: pd.DataFrame):
    labels_df = labels_df.copy()
    labels_df['target_id'] = labels_df['ID'].str.rsplit('_', n=1).str[0]
    labels_df = labels_df.sort_values(['target_id', 'resid'])
    coord_cols = ['x_1', 'y_1', 'z_1']
    arr = labels_df[coord_cols].values.copy()
    arr[arr < -1e6] = np.nan
    labels_df[coord_cols] = arr
    coords_dict = {}
    for tid, grp in tqdm(labels_df.groupby('target_id', sort=False),
                         desc="Processing structures"):
        coords_dict[tid] = grp[coord_cols].values
    return coords_dict

train_coords_dict = process_labels(train_labels)
valid_coords_dict = process_labels(validation_labels)

# ─────────────────────────────────────────────────────────────
# SECTION 4 — Enhanced TBM: aligners, features, search
# ─────────────────────────────────────────────────────────────

def create_global_aligner():
    a = PairwiseAligner(); a.mode = "global"
    a.match_score = 2; a.mismatch_score = -1.5
    a.open_gap_score = -8; a.extend_gap_score = -0.4
    for attr in ["query_left","query_right","target_left","target_right"]:
        setattr(a, f"{attr}_open_gap_score",  -8)
        setattr(a, f"{attr}_extend_gap_score", -0.4)
    return a

def create_local_aligner():
    a = PairwiseAligner(); a.mode = "local"
    a.match_score = 2; a.mismatch_score = -1.5
    a.open_gap_score = -6; a.extend_gap_score = -0.3
    return a

global_aligner = create_global_aligner()
local_aligner  = create_local_aligner()

def compute_gc_content(seq):
    s = seq.upper()
    return (s.count('G') + s.count('C')) / len(s) if s else 0.0

def compute_dinucleotide_frequencies(seq):
    s = seq.upper(); nucs = ['A','C','G','U']
    dinucs = [a+b for a in nucs for b in nucs]
    counts = Counter(s[i:i+2] for i in range(len(s)-1) if s[i:i+2] in dinucs)
    v = np.array([counts.get(d, 0) for d in dinucs], dtype=float)
    t = v.sum()
    return v / t if t > 0 else v

def compute_sequence_entropy(seq):
    counts = Counter(seq.upper()); total = len(seq)
    if not total: return 0.0
    return -sum((c/total)*np.log2(c/total) for c in counts.values() if c > 0)

def compute_kmer_overlap(s1, s2, k=3):
    if len(s1) < k or len(s2) < k: return 0.0
    k1 = set(s1[i:i+k] for i in range(len(s1)-k+1))
    k2 = set(s2[i:i+k] for i in range(len(s2)-k+1))
    inter, union = len(k1 & k2), len(k1 | k2)
    return inter / union if union else 0.0

def compute_feature_similarity(q, t):
    gc_sim    = 1.0 - abs(compute_gc_content(q) - compute_gc_content(t))
    d1, d2    = compute_dinucleotide_frequencies(q), compute_dinucleotide_frequencies(t)
    n1, n2    = np.linalg.norm(d1), np.linalg.norm(d2)
    dinuc_sim = np.dot(d1, d2) / (n1*n2) if n1 > 0 and n2 > 0 else 0.0
    ent_sim   = 1.0 - abs(compute_sequence_entropy(q) - compute_sequence_entropy(t)) / 2.0
    return (gc_sim + dinuc_sim + ent_sim) / 3.0

def compute_composite_score(q, t):
    ml = min(len(q), len(t))
    if ml == 0:
        return {'composite_score': 0.0, 'global_score': 0.0,
                'local_score': 0.0, 'feature_score': 0.0, 'kmer_score': 0.0}
    g_n = max(0.0, min(1.0, global_aligner.score(q, t) / (2*ml)))
    l_n = max(0.0, min(1.0, local_aligner.score(q, t)  / (2*ml)))
    f_s = compute_feature_similarity(q, t)
    k_s = compute_kmer_overlap(q, t, k=3)
    comp = 0.4*g_n+0.2*l_n+0.25*f_s+0.1*k_s
    return {'global_score': g_n, 'local_score': l_n,
            'feature_score': f_s, 'kmer_score': k_s, 'composite_score': comp}

def extract_template_features(seq):
    return np.concatenate([[compute_gc_content(seq)],
                           compute_dinucleotide_frequencies(seq),
                           [compute_sequence_entropy(seq)],
                           [np.log10(len(seq)+1)]])

def cluster_and_diversify(candidates, n_final, distance_threshold=0.3):
    if len(candidates) <= n_final: return candidates
    feats = np.array([extract_template_features(c[1]) for c in candidates])
    feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)
    try:
        n_clus = min(len(candidates), max(n_final, int(len(candidates)*0.3)))
        labels = fcluster(linkage(pdist(feats,'euclidean'), 'ward'), n_clus, 'maxclust')
    except Exception:
        labels = np.arange(len(candidates)) + 1
    reps = {}
    for idx, lab in enumerate(labels):
        if lab not in reps or candidates[idx][2] > reps[lab][2]:
            reps[lab] = candidates[idx]
    return sorted(reps.values(), key=lambda x: x[2], reverse=True)[:n_final]

def get_adaptive_length_threshold(qlen):
    return 0.60 if qlen < 50 else (0.20 if qlen > 1000 else 0.40)

def passes_length_filter(qlen, tlen):
    return abs(tlen - qlen) / max(tlen, qlen) <= get_adaptive_length_threshold(qlen)

def find_similar_sequences_enhanced(query_seq, train_seqs_df, train_coords_dict,
                                    top_n=5, candidate_pool_size=30,
                                    enable_clustering=True, temporal_cutoff=None):
    """Returns (candidates_list, best_score_dict)."""
    qlen, candidates = len(query_seq), []
    filtered = (train_seqs_df[train_seqs_df['temporal_cutoff'] < temporal_cutoff]
                if temporal_cutoff else train_seqs_df)
    for _, row in filtered.iterrows():
        tid, tseq = row["target_id"], row["sequence"]
        if tid not in train_coords_dict: continue
        if not passes_length_filter(qlen, len(tseq)): continue
        sd = compute_composite_score(query_seq, tseq)
        candidates.append((tid, tseq, sd['composite_score'], train_coords_dict[tid], sd))

    candidates.sort(key=lambda x: x[2], reverse=True)
    candidates = candidates[:candidate_pool_size]

    if enable_clustering and len(candidates) > top_n:
        slim     = [(c[0],c[1],c[2],c[3]) for c in candidates]
        sdlookup = {c[0]: c[4] for c in candidates}
        slim_div = cluster_and_diversify(slim, n_final=top_n)
        final    = [(c[0],c[1],c[2],c[3],sdlookup.get(c[0],{})) for c in slim_div]
    else:
        final = candidates[:top_n]

    best_scores = final[0][4] if final else {}
    return [(c[0],c[1],c[2],c[3]) for c in final], best_scores

# ─────────────────────────────────────────────────────────────
# SECTION 5 — Template transfer + geometry refinement
# ─────────────────────────────────────────────────────────────

def adapt_template_to_query(query_seq, template_seq, template_coords):
    alignment  = next(iter(global_aligner.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan, dtype=float)
    for (q_s,q_e),(t_s,t_e) in zip(*alignment.aligned):
        chunk = template_coords[t_s:t_e]
        if len(chunk) == (q_e - q_s):
            new_coords[q_s:q_e] = chunk
    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            pv = next((j for j in range(i-1,-1,-1) if not np.isnan(new_coords[j,0])), -1)
            nv = next((j for j in range(i+1,len(new_coords)) if not np.isnan(new_coords[j,0])), -1)
            if pv >= 0 and nv >= 0:
                w = (i-pv)/(nv-pv)
                new_coords[i] = (1-w)*new_coords[pv] + w*new_coords[nv]
            elif pv >= 0: new_coords[i] = new_coords[pv] + [3,0,0]
            elif nv >= 0: new_coords[i] = new_coords[nv] + [3,0,0]
            else:         new_coords[i] = [i*3, 0, 0]
    return np.nan_to_num(new_coords)

def adaptive_rna_constraints(coordinates, target_id_or_segs, confidence=1.0, passes=2):
    """Physical constraint refinement. Accepts either target_id string or segments list."""
    coords = coordinates.copy()
    if isinstance(target_id_or_segs, str):
        segments = test_segs_map.get(target_id_or_segs, [(0, len(coords))])
    else:
        segments = target_id_or_segs if target_id_or_segs else [(0, len(coords))]
    strength = max(0.02, 0.75*(1.0 - min(confidence, 0.90)))
    for _ in range(passes):
        for (s, e) in segments:
            X = coords[s:e]; L = e - s
            if L < 3: coords[s:e] = X; continue
            d    = X[1:]-X[:-1]; dist = np.linalg.norm(d,axis=1)+1e-6
            adj  = (d*(5.95-dist)[:,None]/dist[:,None])*(0.22*strength)
            X[:-1] -= adj; X[1:] += adj
            d2   = X[2:]-X[:-2]; dist2 = np.linalg.norm(d2,axis=1)+1e-6
            adj2 = (d2*(10.2-dist2)[:,None]/dist2[:,None])*(0.10*strength)
            X[:-2] -= adj2; X[2:] += adj2
            lap  = 0.5*(X[:-2]+X[2:])-X[1:-1]
            X[1:-1] += (0.06*strength)*lap
            if L >= 25:
                k   = min(L,160) if L > 220 else L
                idx = np.linspace(0,L-1,k).astype(int) if k < L else np.arange(L)
                P   = X[idx]; diff = P[:,None,:]-P[None,:,:]
                dm  = np.linalg.norm(diff,axis=2)+1e-6
                sep = np.abs(idx[:,None]-idx[None,:])
                mask = (sep>2)&(dm<3.2)
                if np.any(mask):
                    force = (3.2-dm)/dm
                    vec   = (diff*force[:,:,None]*mask[:,:,None]).sum(axis=1)
                    X[idx] += (0.015*strength)*vec
            coords[s:e] = X
    return coords

# ─────────────────────────────────────────────────────────────
# SECTION 6 — Geometric transform helpers
# ─────────────────────────────────────────────────────────────

def _rotmat(axis, ang):
    axis = np.asarray(axis, float)
    axis = axis / (np.linalg.norm(axis)+1e-12)
    x,y,z = axis; c,s = np.cos(ang),np.sin(ang); C = 1-c
    return np.array([[c+x*x*C,    x*y*C-z*s, x*z*C+y*s],
                     [y*x*C+z*s, c+y*y*C,    y*z*C-x*s],
                     [z*x*C-y*s, z*y*C+x*s, c+z*z*C   ]], float)

def apply_hinge(coords, seg, rng, max_angle_deg=25):
    s, e = seg; L = e - s
    if L < 30: return coords
    pivot = s + int(rng.integers(10, L-10))
    Rmat  = _rotmat(rng.normal(size=3),
                    np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg))))
    X = coords.copy(); p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e]-p0) @ Rmat.T + p0
    return X

def jitter_chains(coords, segments, rng, max_angle_deg=15, max_trans=1.5):
    X = coords.copy(); gc = X.mean(0, keepdims=True)
    for (s,e) in segments:
        Rmat = _rotmat(rng.normal(size=3),
                       np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg))))
        sh   = rng.normal(size=3)
        sh   = sh/(np.linalg.norm(sh)+1e-12)*float(rng.uniform(0, max_trans))
        c    = X[s:e].mean(0, keepdims=True)
        X[s:e] = (X[s:e]-c) @ Rmat.T + c + sh
    X -= X.mean(0, keepdims=True) - gc
    return X

def smooth_wiggle(coords, segments, rng, amp=0.8):
    X = coords.copy()
    for (s,e) in segments:
        L = e - s
        if L < 25: continue
        ctrl_x = np.linspace(0, L-1, 6)
        ctrl_d = rng.normal(0, amp, (6,3))
        t      = np.arange(L)
        X[s:e] += np.vstack([np.interp(t, ctrl_x, ctrl_d[:,k]) for k in range(3)]).T
    return X

def centroid_align_and_blend(arr1, arr2, w1, w2):
    """Blend two coordinate sets after centroid-aligning each."""
    c1, c2  = arr1.mean(axis=0), arr2.mean(axis=0)
    blended = w1*(arr1-c1) + w2*(arr2-c2)
    # restore the Boltz/secondary centroid position
    blended += w1*c1 + w2*c2
    return blended

# ─────────────────────────────────────────────────────────────
# SECTION 7 — Build enhanced TBM predictions (5 per target)
# ─────────────────────────────────────────────────────────────

target_score_log  = {}   # { target_id: score_dict }
template_preds_dict = {}  # { target_id: [pred_1 .. pred_5] }

def build_template_predictions(row, n_predictions=5):
    tid      = row["target_id"]
    seq      = row["sequence"]
    segments = test_segs_map.get(tid, [(0, len(seq))])
    cutoff   = row.get('temporal_cutoff', None)

    cands, best_scores = find_similar_sequences_enhanced(
        query_seq=seq, train_seqs_df=train_seqs,
        train_coords_dict=train_coords_dict,
        top_n=30, candidate_pool_size=50,
        enable_clustering=True, temporal_cutoff=cutoff)

    target_score_log[tid] = {k: round(v,4) for k,v in best_scores.items()}

    predictions, used = [], set()
    for i in range(n_predictions):
        seed_val = (row.name * 10_000_000_000 + i * 10007) % (2**32)
        rng = np.random.default_rng(seed_val)

        if not cands:
            coords = np.zeros((len(seq), 3), float)
            for (s,e) in segments:
                for j in range(s+1, e): coords[j] = coords[j-1] + [5.95,0,0]
            predictions.append(coords); continue

        if i == 0:
            t_id, t_seq, sim, t_coords = cands[0]
        else:
            K    = min(12, len(cands))
            sims = np.array([cands[k][2] for k in range(K)], float)
            w    = np.exp((sims - sims.max()) / 0.08)
            for k in range(K):
                if cands[k][0] in used: w[k] *= 0.10
            w    = w / (w.sum()+1e-12)
            k    = int(rng.choice(np.arange(K), p=w))
            t_id, t_seq, sim, t_coords = cands[k]
        used.add(t_id)

        adapted = adapt_template_to_query(seq, t_seq, t_coords)

        if   i == 0: X = adapted
        elif i == 1: X = adapted + rng.normal(0, max(0.01,(0.40-sim)*0.06), adapted.shape)
        elif i == 2:
            longest = max(segments, key=lambda se: se[1]-se[0])
            X = apply_hinge(adapted, longest, rng, max_angle_deg=22)
        elif i == 3: X = jitter_chains(adapted, segments, rng, 10, 1.0)
        else:        X = smooth_wiggle(adapted, segments, rng, 0.7)

        refined = adaptive_rna_constraints(X, tid, confidence=sim, passes=2)
        predictions.append(refined)

    return predictions

print("Building enhanced TBM predictions for all test targets...")
for idx, row in tqdm(test_seqs.iterrows(), total=len(test_seqs), desc="TBM"):
    template_preds_dict[row["target_id"]] = build_template_predictions(row)

# Also save TBM predictions as CSV (needed as RNAPro templates)
tbm_rows = []
for idx, row in test_seqs.iterrows():
    tid  = row["target_id"]
    seq  = row["sequence"]
    preds = template_preds_dict[tid]
    for j in range(len(seq)):
        r = {"ID": f"{tid}_{j+1}", "resname": seq[j], "resid": j+1}
        for i in range(5):
            r[f"x_{i+1}"] = preds[i][j][0]
            r[f"y_{i+1}"] = preds[i][j][1]
            r[f"z_{i+1}"] = preds[i][j][2]
        tbm_rows.append(r)

sub_tbm = pd.DataFrame(tbm_rows)
col_order = ["ID","resname","resid"] + \
            [f"{c}_{i}" for i in range(1,6) for c in ["x","y","z"]]
sub_tbm = sub_tbm[col_order]
sub_tbm.to_csv("/kaggle/working/submission_tbm.csv", index=False)
print("✓ submission_tbm.csv saved")

# ─────────────────────────────────────────────────────────────
# SECTION 8 — RNAPro inference
# ─────────────────────────────────────────────────────────────

# Convert enhanced TBM CSV to .pt template file for RNAPro
os.chdir("RNAPro")
os.system(
    "python preprocess/convert_templates_to_pt_files.py "
    "--input_csv /kaggle/working/submission_tbm.csv "
    "--output_name templates.pt"
)
os.chdir("..")

# Copy required CIF reference files
DIST = "/kaggle/working/RNAPro/release_data/ccd_cache/"
os.makedirs(DIST, exist_ok=True)
os.system(f"cp /kaggle/input/protenix-checkpoints/components.v20240608.cif {DIST}")
os.system(f"cp /kaggle/input/protenix-checkpoints/components.v20240608.cif.rdkit_mol.pkl {DIST}")

# Write the sequences to process (full set in scoring run, else head-5)
df_seq = pd.read_csv(f'{BASE_PATH}/test_sequences.csv')
if not IS_SCORING_RUN:
    df_seq = df_seq.head(5)
df_seq.to_csv('/kaggle/working/sample_sequences.csv', index=False)

# Write the RNAPro inference runner script
rnapro_runner_code = r"""
import os, shutil, logging, traceback, warnings, argparse, json
from contextlib import nullcontext
from os.path import join as opjoin
from typing import Any, Mapping
import torch
import pandas as pd
import numpy as np
from biotite.structure.io import pdbx

from configs.configs_base import configs as configs_base
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs
from runner.dumper import DataDumper
from rnapro.config import parse_sys_args
from rnapro.config.config import ConfigManager, ArgumentNotSet
from rnapro.data.infer_data_pipeline import get_inference_dataloader
from rnapro.model.RNAPro import RNAPro
from rnapro.utils.distributed import DIST_WRAPPER
from rnapro.utils.seed import seed_everything
from rnapro.utils.torch_utils import to_device

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.basicConfig(level=logging.WARNING)
logging.getLogger("rnapro").setLevel(logging.WARNING)

def parse_configs(configs, arg_str=None, fill_required_with_null=False):
    manager = ConfigManager(configs, fill_required_with_null=fill_required_with_null)
    parser = argparse.ArgumentParser()
    parser.add_argument("--max_len", type=int, default=10000, required=False)
    for key, (dtype, default_value, allow_none, required) in manager.config_infos.items():
        parser.add_argument("--" + key, type=str, default=ArgumentNotSet(), required=required)
    merged_configs = manager.merge_configs(
        vars(parser.parse_args(arg_str.split())) if arg_str else {})
    merged_configs.max_len = parser.parse_args(arg_str.split()).max_len
    return merged_configs

class dotdict(dict):
    __setattr__ = dict.__setitem__; __delattr__ = dict.__delitem__
    def __getattr__(self, name):
        try: return self[name]
        except KeyError: raise AttributeError(name)

class InferenceRunner:
    def __init__(self, configs):
        self.configs = configs
        self.init_env(); self.init_basics(); self.init_model()
        self.load_checkpoint()
        self.init_dumper(need_atom_confidence=configs.need_atom_confidence,
                         sorted_by_ranking_score=configs.sorted_by_ranking_score)
    def init_env(self):
        self.use_cuda = torch.cuda.device_count() > 0
        self.device = torch.device("cuda:{}".format(DIST_WRAPPER.local_rank) if self.use_cuda else "cpu")
        if self.use_cuda: torch.cuda.set_device(self.device)
    def init_basics(self):
        self.dump_dir = self.configs.dump_dir; self.error_dir = opjoin(self.dump_dir, "ERR")
        os.makedirs(self.dump_dir, exist_ok=True); os.makedirs(self.error_dir, exist_ok=True)
    def init_model(self):
        self.model = RNAPro(self.configs).to(self.device)
    def load_checkpoint(self):
        ckpt_path = self.configs.load_checkpoint_path
        if not os.path.exists(ckpt_path): raise Exception(f"Checkpoint missing: {ckpt_path}")
        ckpt = torch.load(ckpt_path, self.device)
        if list(ckpt["model"].keys())[0].startswith("module."):
            ckpt["model"] = {k[7:]: v for k,v in ckpt["model"].items()}
        self.model.load_state_dict(state_dict=ckpt["model"], strict=True)
        self.model.eval()
    def init_dumper(self, need_atom_confidence=False, sorted_by_ranking_score=True):
        self.dumper = DataDumper(base_dir=self.dump_dir,
                                 need_atom_confidence=need_atom_confidence,
                                 sorted_by_ranking_score=sorted_by_ranking_score)
    @torch.no_grad()
    def predict(self, data):
        prec = {"fp32": torch.float32, "bf16": torch.bfloat16, "fp16": torch.float16}[self.configs.dtype]
        ctx = torch.autocast(device_type="cuda", dtype=prec) if torch.cuda.is_available() else nullcontext()
        data = to_device(data, self.device)
        with ctx:
            prediction, _, _ = self.model(input_feature_dict=data["input_feature_dict"],
                                          label_full_dict=None, label_dict=None, mode="inference")
        return prediction
    def print(self, msg):
        if DIST_WRAPPER.rank == 0: print(msg)
    def update_model_configs(self, new_configs):
        self.model.configs = new_configs

def update_inference_configs(configs, N_token):
    if N_token > 3840:   configs.skip_amp.confidence_head = False; configs.skip_amp.sample_diffusion = False
    elif N_token > 2560: configs.skip_amp.confidence_head = False; configs.skip_amp.sample_diffusion = True
    else:                configs.skip_amp.confidence_head = True;  configs.skip_amp.sample_diffusion = True
    return configs

def infer_predict(runner, configs):
    try: dataloader = get_inference_dataloader(configs=configs)
    except Exception as e:
        with open(opjoin(runner.error_dir, "error.txt"), "a") as f: f.write(f"{e}\n{traceback.format_exc()}")
        return
    num_data = len(dataloader.dataset)
    for seed in configs.seeds:
        seed_everything(seed=seed, deterministic=configs.deterministic)
        for batch in dataloader:
            try:
                data, atom_array, data_error_message = batch[0]
                sample_name = data["sample_name"]
                if len(data_error_message) > 0:
                    with open(opjoin(runner.error_dir, f"{sample_name}.txt"), "a") as f: f.write(data_error_message)
                    continue
                new_configs = update_inference_configs(configs, data["N_token"].item())
                runner.update_model_configs(new_configs)
                prediction = runner.predict(data)
                runner.dumper.dump(dataset_name="", pdb_id=sample_name, seed=seed,
                                   pred_dict=prediction, atom_array=atom_array,
                                   entity_poly_type=data["entity_poly_type"])
                torch.cuda.empty_cache()
            except Exception as e:
                with open(opjoin(runner.error_dir, f"{sample_name}.txt"), "a") as f:
                    f.write(f"{e}\n{traceback.format_exc()}")
                if hasattr(torch.cuda, "empty_cache"): torch.cuda.empty_cache()

def make_dummy_solution(df):
    sol = dotdict()
    for _, r in df.iterrows():
        sol[r.target_id] = dotdict(target_id=r.target_id, sequence=r.sequence, coord=[])
    return sol

def solution_to_submit_df(solution):
    frames = []
    for k, s in solution.items():
        L = len(s.sequence)
        df = pd.DataFrame()
        df["ID"] = [f"{s.target_id}_{i+1}" for i in range(L)]
        df["resname"] = list(s.sequence); df["resid"] = list(range(1, L+1))
        for j, c in enumerate(s.coord):
            df[f"x_{j+1}"] = c[:,0]; df[f"y_{j+1}"] = c[:,1]; df[f"z_{j+1}"] = c[:,2]
        frames.append(df)
    return pd.concat(frames)

def extract_c1_coordinates(cif_file_path):
    try:
        with open(cif_file_path, "r") as f: cif_data = pdbx.CIFFile.read(f)
        atom_array = pdbx.get_structure(cif_data, model=1)
        mask  = np.char.strip(atom_array.atom_name.astype(str)) == "C1'"
        c1    = atom_array[mask]
        if len(c1) == 0: return None
        idx   = np.argsort(c1.res_id)
        return c1[idx].coord
    except Exception as e:
        print(f"Error extracting C1' from {cif_file_path}: {e}"); return None

def create_input_json(sequence, target_id):
    return [{"sequences": [{"rnaSequence": {"sequence": sequence, "count": 1}}], "name": target_id}]

def run_ptx(target_id, sequence, configs, solution, template_idx, runner):
    temp_dir = f"./{configs.dump_dir}/input"; output_dir = f"./{configs.dump_dir}/output"
    os.makedirs(temp_dir, exist_ok=True); os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(temp_dir, f"{target_id}_input.json")
    with open(json_path, "w") as f: json.dump(create_input_json(sequence, target_id), f)
    configs.input_json_path = json_path; configs.template_idx = int(template_idx)
    infer_predict(runner, configs)
    cif_path = f"{configs.dump_dir}/{target_id}/seed_42/predictions/{target_id}_sample_0.cif"
    coord = extract_c1_coordinates(cif_path)
    if coord is None: coord = np.zeros((len(sequence), 3), np.float32)
    elif coord.shape[0] < len(sequence):
        coord = np.concatenate([coord, np.zeros((len(sequence)-coord.shape[0], 3), np.float32)])
    solution[target_id].coord.append(coord)

def run():
    configs_base["use_deepspeed_evo_attention"] = \
        os.environ.get("USE_DEEPSPEED_EVO_ATTENTION", False) == "true"
    configs = {**configs_base, **{"data": data_configs}, **inference_configs}
    configs = parse_configs(configs=configs, arg_str=parse_sys_args(), fill_required_with_null=True)
    valid_df = pd.read_csv(configs.sequences_csv)
    print(f"\n -> {len(valid_df)} sequence(s) to process")
    runner  = InferenceRunner(configs)
    solution = make_dummy_solution(valid_df)
    for idx, row in valid_df.iterrows():
        print(f"\n -> {row.target_id} (len={len(row.sequence)})")
        if len(row.sequence) > configs.max_len:
            print(f"    Skipping — too long (> {configs.max_len})")
            for _ in range(5): solution[row.target_id].coord.append(np.zeros((len(row.sequence),3), np.float32))
            continue
        try:
            for template_idx in range(5):
                run_ptx(row.target_id, row.sequence, configs, solution, template_idx, runner)
        except Exception as e:
            print(f"    ERROR: {e}")
    submit_df = solution_to_submit_df(solution).fillna(0.0)
    submit_df.to_csv("./submission_rnapro.csv", index=False)
    print("\n -> Saved submission_rnapro.csv")

if __name__ == "__main__":
    run()
"""
with open("RNAPro/runner/inference.py", "w") as f:
    f.write(rnapro_runner_code)

# Write the bash launcher
rnapro_sh = """#!/usr/bin/env bash
export LAYERNORM_TYPE=torch

SEED=42
N_SAMPLE=1
N_STEP=200
N_CYCLE=10

DUMP_DIR="../rnapro_output"
CHECKPOINT_PATH="../rnapro-private-best-500m.ckpt"
TEMPLATE_DATA="./release_data/kaggle/templates.pt"
RNA_MSA_DIR="/kaggle/input/stanford-rna-3d-folding-2/MSA"
SEQUENCES_CSV="/kaggle/working/sample_sequences.csv"
RIBONANZA_PATH="/kaggle/input/ribonanzanet2/pytorch/alpha/1/"
MODEL_NAME="rnapro_base"

mkdir -p "${DUMP_DIR}"

python3 runner/inference.py \\
    --model_name "${MODEL_NAME}" \\
    --seeds ${SEED} \\
    --dump_dir "${DUMP_DIR}" \\
    --load_checkpoint_path "${CHECKPOINT_PATH}" \\
    --use_msa true \\
    --use_template "ca_precomputed" \\
    --model.use_template "ca_precomputed" \\
    --model.use_RibonanzaNet2 true \\
    --model.template_embedder.n_blocks 2 \\
    --model.ribonanza_net_path "${RIBONANZA_PATH}" \\
    --template_data "${TEMPLATE_DATA}" \\
    --template_idx 0 \\
    --rna_msa_dir "${RNA_MSA_DIR}" \\
    --model.N_cycle ${N_CYCLE} \\
    --sample_diffusion.N_sample ${N_SAMPLE} \\
    --sample_diffusion.N_step ${N_STEP} \\
    --load_strict true \\
    --num_workers 0 \\
    --triangle_attention "torch" \\
    --triangle_multiplicative "torch" \\
    --sequences_csv "${SEQUENCES_CSV}" \\
    --max_len 1000
"""
with open("RNAPro/rnapro_inference_kaggle.sh", "w") as f:
    f.write(rnapro_sh)

os.chdir("RNAPro")
os.system("bash ./rnapro_inference_kaggle.sh")
os.chdir("..")
os.system("cp rnapro_output/submission_rnapro.csv /kaggle/working/submission_rnapro.csv")
print("✓ RNAPro inference done")

# ─────────────────────────────────────────────────────────────
# SECTION 9 — Collect RNAPro coordinate arrays
# ─────────────────────────────────────────────────────────────

# rnapro_preds_dict[target_id] = list of up to 5 (L,3) arrays (one per template_idx)
rnapro_preds_dict = {}

try:
    df_rnapro = pd.read_csv("/kaggle/working/submission_rnapro.csv")
    df_rnapro["target_id"] = df_rnapro["ID"].str.rsplit("_", n=1).str[0]

    for tid, grp in df_rnapro.groupby("target_id", sort=False):
        grp = grp.sort_values("resid")
        preds = []
        for k in range(1, 6):
            cols = [f"x_{k}", f"y_{k}", f"z_{k}"]
            if all(c in grp.columns for c in cols):
                arr = grp[cols].values.astype(float)
                if np.isfinite(arr).all() and not np.all(arr == 0):
                    preds.append(arr)
        rnapro_preds_dict[tid] = preds if preds else []

    n_rnapro = sum(1 for v in rnapro_preds_dict.values() if v)
    print(f"✓ RNAPro predictions collected for {n_rnapro}/{len(test_seqs)} targets")
except FileNotFoundError:
    print("⚠  submission_rnapro.csv not found — RNAPro predictions will be unavailable")

# ─────────────────────────────────────────────────────────────
# SECTION 10 — Boltz inference
# ─────────────────────────────────────────────────────────────

max_boltz_tokens = 900
pred_repeats     = 5

os.system("mkdir -p input_fasta")
os.system("rm -rf boltz_results*")

print("\n" + "="*60 + "\nRUNNING BOLTZ PREDICTIONS\n" + "="*60)

for k in range(len(test_seqs)):
    row  = test_seqs.iloc[k]
    name = row.target_id

    if len(row.sequence) > max_boltz_tokens:
        print(f"  [{k+1}] {name}: SKIPPED (len={len(row.sequence)} > {max_boltz_tokens})")
        continue

    print(f"\n[{k+1}/{len(test_seqs)}] Preparing Boltz input for {name}")

    pred_seq = row.sequence
    try:
        chains, chain_ind = {}, 1
        repeats = 1
        for entry in row.all_sequences.split("\n"):
            entry = entry.strip()
            if not entry: continue
            if entry.startswith(">"):
                try:   repeats = min(len(entry.split("|")[1].split(",")), 999)
                except: repeats = 1
            else:
                if entry == pred_seq.strip():
                    chains[0] = (pred_seq, "rna")
        if 0 not in chains:
            chains[0] = (pred_seq, "rna")
    except Exception:
        chains = {0: (pred_seq, "rna")}

    with open(f"input_fasta/{name}.fasta", "w") as fout:
        for chain in sorted(chains.keys()):
            seq_c, seq_type = chains[chain]
            msa = "" if seq_type == "rna" else "empty"
            fout.write(f">{chain}|{seq_type}|{msa}\n{seq_c}\n")

    for repeat in range(pred_repeats):
        os.system(
            f"boltz predict input_fasta/{name}.fasta "
            f"--num_workers 1 --max_parallel_samples 1 "
            f"--output_format pdb "
            f"--cache boltz_cache/boltz2/ "
            f"--out_dir boltz_repeat_{repeat}"
        )
        gc.collect(); torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────────
# SECTION 11 — Collect Boltz C1' coordinates
# ─────────────────────────────────────────────────────────────

from biopandas.pdb import PandasPdb

print("\n" + "="*60 + "\nCOLLECTING BOLTZ PREDICTIONS\n" + "="*60)

boltz_preds_dict = {}  # { target_id: [repeat_0, ..., repeat_4] }  each entry (L,3) or None

for k in range(len(test_seqs)):
    row   = test_seqs.iloc[k]
    name  = row.target_id
    L_seq = len(row.sequence)
    repeat_coords = []
    for rep in range(pred_repeats):
        fname = (f"boltz_repeat_{rep}/boltz_results_{name}"
                 f"/predictions/{name}/{name}_model_0.pdb")
        try:
            atom_df  = PandasPdb().read_pdb(fname).df["ATOM"]
            c1_atoms = (atom_df[(atom_df.chain_id == "0") & (atom_df.atom_name == "C1'")]
                        .sort_values("residue_number"))
            if len(c1_atoms) != L_seq:
                repeat_coords.append(None); continue
            repeat_coords.append(c1_atoms[["x_coord","y_coord","z_coord"]].values.astype(float))
        except Exception:
            repeat_coords.append(None)

    boltz_preds_dict[name] = repeat_coords
    n_ok = sum(r is not None for r in repeat_coords)
    print(f"  {name}: {n_ok}/{pred_repeats} Boltz repeats")

# ─────────────────────────────────────────────────────────────
# SECTION 12 — Tri-model hybrid combination (5 predictions)
# ─────────────────────────────────────────────────────────────

RNAPRO_MAX_LEN = 1000  # RNAPro was only run for sequences ≤ this length

def first_valid(lst):
    """Return first non-None element, or None."""
    for x in lst:
        if x is not None: return x
    return None

def build_hybrid_predictions(row, tmpl_preds, boltz_repeats, rnapro_preds):
    """
    Combine up to three model sources into exactly 5 predictions.

    Priority scheme:
      Pred 1 — best TBM            (most trusted template model)
      Pred 2 — best Boltz repeat   (Boltz repeat-0, or first valid)
      Pred 3 — best RNAPro         (RNAPro template_idx-0)
      Pred 4 — Boltz / TBM 50-50 blend (centroid-aligned)
      Pred 5 — RNAPro / Boltz 50-50 blend  (or RNAPro / TBM if no Boltz)

    All predictions with a DL source are refined with adaptive_rna_constraints.
    When a DL model is unavailable for a target, geometric TBM variants fill in.
    """
    tid      = row["target_id"]
    seq      = row["sequence"]
    segments = test_segs_map.get(tid, [(0, len(seq))])
    seed_val = (row.name * 99_999 + 42) % (2**32)
    rng      = np.random.default_rng(seed_val)
    longest  = max(segments, key=lambda se: se[1]-se[0])

    tmpl       = tmpl_preds[0]                   # always available
    boltz_arr  = first_valid(boltz_repeats)       # None if sequence too long / failed
    rnapro_arr = first_valid(rnapro_preds) if rnapro_preds else None

    # ── Pred 1: best TBM ──────────────────────────────────────
    pred1 = tmpl.copy()

    # ── Pred 2: best Boltz (or TBM hinge variant) ────────────
    if boltz_arr is not None and boltz_arr.shape == tmpl.shape:
        pred2 = boltz_arr.copy()
    else:
        pred2 = adaptive_rna_constraints(
            apply_hinge(tmpl, longest, rng, max_angle_deg=22), tid, confidence=0.5)

    # ── Pred 3: best RNAPro (or TBM jitter variant) ───────────
    if rnapro_arr is not None and rnapro_arr.shape == tmpl.shape:
        pred3 = rnapro_arr.copy()
    else:
        pred3 = adaptive_rna_constraints(
            jitter_chains(tmpl, segments, rng, 12, 1.5), tid, confidence=0.5)

    # ── Pred 4: 50/50 Boltz + TBM blend ──────────────────────
    if boltz_arr is not None and boltz_arr.shape == tmpl.shape:
        raw4 = centroid_align_and_blend(tmpl, boltz_arr, 0.5, 0.5)
    else:
        # Fallback: smooth wiggle on best TBM
        raw4 = smooth_wiggle(tmpl, segments, rng, amp=0.8)
    pred4 = adaptive_rna_constraints(raw4, tid, confidence=0.75, passes=2)

    # ── Pred 5: 50/50 RNAPro + Boltz blend (or RNAPro + TBM) ─
    if rnapro_arr is not None and rnapro_arr.shape == tmpl.shape:
        partner = boltz_arr if (boltz_arr is not None and boltz_arr.shape == tmpl.shape) else tmpl
        raw5    = centroid_align_and_blend(rnapro_arr, partner, 0.5, 0.5)
        jitter  = rng.normal(0, 0.15, tmpl.shape)
        raw5   += jitter
    else:
        # Fallback: second TBM hinge with different rng state
        raw5 = smooth_wiggle(tmpl, segments, rng, amp=1.2)
    pred5 = adaptive_rna_constraints(raw5, tid, confidence=0.75, passes=2)

    return [pred1, pred2, pred3, pred4, pred5]

# ─────────────────────────────────────────────────────────────
# SECTION 13 — Main prediction loop and submission writer
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60 + "\nASSEMBLING FINAL HYBRID SUBMISSION\n" + "="*60)

all_predictions = []
strategy_rows   = []
start_time      = time.time()

for idx, row in test_seqs.iterrows():
    tid = row["target_id"]
    seq = row["sequence"]
    L   = len(seq)

    if idx % 10 == 0:
        print(f"  {idx}/{len(test_seqs)}  {tid}  | {time.time()-start_time:.1f}s")

    tmpl_preds    = template_preds_dict[tid]                         # always 5
    boltz_repeats = boltz_preds_dict.get(tid, [None]*pred_repeats)   # 5 or all-None
    rnapro_preds  = rnapro_preds_dict.get(tid, [])                   # up to 5 or []

    final_preds = build_hybrid_predictions(row, tmpl_preds, boltz_repeats, rnapro_preds)

    assert len(final_preds) == 5
    for p in final_preds:
        assert isinstance(p, np.ndarray) and p.shape == (L, 3), \
            f"Bad pred shape for {tid}: {getattr(p,'shape',None)}"
        if not np.isfinite(p).all():
            p[:] = np.nan_to_num(p)

    for j in range(L):
        res = {"ID": f"{tid}_{j+1}", "resname": seq[j], "resid": j+1}
        for i in range(5):
            res[f"x_{i+1}"], res[f"y_{i+1}"], res[f"z_{i+1}"] = final_preds[i][j]
        all_predictions.append(res)

    n_boltz  = sum(r is not None for r in boltz_repeats)
    n_rnapro = len(rnapro_preds)
    strategy = ("tri_model"   if n_boltz > 0 and n_rnapro > 0 else
                "boltz_tbm"   if n_boltz > 0 else
                "rnapro_tbm"  if n_rnapro > 0 else
                "tbm_only")
    strategy_rows.append({
        "target_id":     tid,
        "seq_len":       L,
        "boltz_repeats": n_boltz,
        "rnapro_preds":  n_rnapro,
        "strategy":      strategy,
        **{k: round(v,4) for k,v in target_score_log.get(tid,{}).items()},
    })

# ── Build and save submission CSV ────────────────────────────
sub = pd.DataFrame(all_predictions)
cols = ["ID","resname","resid"] + [f"{c}_{i}" for i in range(1,6) for c in ["x","y","z"]]
coord_cols = [c for c in cols if c not in ("ID","resname","resid")]
sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
sub[cols].to_csv("submission.csv", index=False)
print("\n✓ submission.csv saved")

# ── Strategy log ─────────────────────────────────────────────
strategy_df = pd.DataFrame(strategy_rows)
strategy_df.to_csv("strategy_log.csv", index=False)

print("\n===== Strategy Summary =====")
print(strategy_df.groupby("strategy")["seq_len"].agg(["count","mean"]).round(1).rename(
    columns={"count":"targets","mean":"avg_len"}))

# ─────────────────────────────────────────────────────────────
# SECTION 14 — Optional validation scoring
# ─────────────────────────────────────────────────────────────

try:
    import runpy
    module_globals = runpy.run_path("/kaggle/usr/lib/tm-score-permutechains/metric.py")
    score = module_globals['score']

    sol = pd.read_csv(f'{BASE_PATH}/validation_labels.csv')
    sub_val = pd.read_csv('/kaggle/working/submission.csv')
    sol['target_id']     = sol['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))
    sub_val['target_id'] = sub_val['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))

    results_rows = []
    for target_id, group_native in sol.groupby('target_id'):
        group_pred = sub_val[sub_val['target_id'] == target_id]
        tm_result  = score(group_native, group_pred, 'ID')
        comp       = target_score_log.get(target_id, {})
        strat      = strategy_df[strategy_df.target_id == target_id]['strategy'].values
        results_rows.append({
            'target_id':    target_id,
            'strategy':     strat[0] if len(strat) else 'unknown',
            'global_score': comp.get('global_score', float('nan')),
            'tm_score':     round(float(tm_result), 5),
        })

    results_df = (pd.DataFrame(results_rows)
                  .sort_values('tm_score', ascending=False)
                  .reset_index(drop=True))
    summary    = pd.DataFrame([{
        'target_id': 'MEAN', 'strategy': '',
        'global_score': round(results_df['global_score'].mean(), 4),
        'tm_score':     round(results_df['tm_score'].mean(), 5),
    }])
    display_df = pd.concat([results_df, summary], ignore_index=True)
    display_df.to_csv('/kaggle/working/score_table.csv', index=False)

    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.max_rows', 200)
    print("\n===== Per-Target Score Breakdown =====")
    print(display_df.to_string(index=False))
    print(f"\nMean TM-score: {results_df['tm_score'].mean():.5f}  (n={len(results_df)})")
    print("\n===== By Strategy =====")
    print(results_df.groupby('strategy')['tm_score'].agg(['mean','count']).round(5))

except Exception as e:
    print(f"\nValidation scoring skipped: {e}")

IS_SCORING_RUN = None
Processing /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
  Attempting uninstall: biopython
    Found existing installation: biopython 1.84
    Uninstalling biopython-1.84:
      Successfully uninstalled biopython-1.84
Obtaining file:///kaggle/working/boltz-src-minimal
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for boltz (pyproject.toml): started
  Building editable for boltz (pyproject.toml): finished with status 'done'
  Created wheel for boltz: filename=boltz-2.2.1-0.editable-py3-none-any.whl size=5248 sha256=a7c87babb64c649c99a567acba9de3263e5ff6bae7ba37a149fa3231c1f256f1
  Stored in directory: /tmp/pip-ephem-wheel-cac

Processing structures:   0%|          | 0/5716 [00:00<?, ?it/s]

Processing structures:   0%|          | 0/28 [00:00<?, ?it/s]

Building enhanced TBM predictions for all test targets...


TBM:   0%|          | 0/28 [00:00<?, ?it/s]

...

100%|██████████| 1/1 [00:00<00:00, 17.75it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-02-16 06:35:52.790948: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771223752.808722    3589 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771223752.813951    3589 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:17

Predicting DataLoader 0: 100%|██████████| 1/1 [00:10<00:00,  0.10it/s]
Checking input data.
Processing 1 inputs with 1 threads.
Running structure prediction for 1 input.


100%|██████████| 1/1 [00:00<00:00, 19.63it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-02-16 06:36:49.983295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771223810.000983    3615 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771223810.006174    3615 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:17

Predicting DataLoader 0: 100%|██████████| 1/1 [00:10<00:00,  0.10it/s]
  [28] 9ZCC: SKIPPED (len=1460 > 900)

COLLECTING BOLTZ PREDICTIONS
  8ZNQ: 5/5 Boltz repeats
  9IWF: 5/5 Boltz repeats
  9JGM: 5/5 Boltz repeats
  9MME: 0/5 Boltz repeats
  9J09: 5/5 Boltz repeats
  9E9Q: 5/5 Boltz repeats
  9CFN: 5/5 Boltz repeats
  9OBM: 5/5 Boltz repeats
  9G4P: 5/5 Boltz repeats
  9G4Q: 5/5 Boltz repeats
  9G4R: 5/5 Boltz repeats
  9RVP: 5/5 Boltz repeats
  9JFS: 5/5 Boltz repeats
  9LEC: 5/5 Boltz repeats
  9LEL: 5/5 Boltz repeats
  9I9W: 5/5 Boltz repeats
  9HRO: 5/5 Boltz repeats
  9QZJ: 5/5 Boltz repeats
  9JFO: 5/5 Boltz repeats
  9OD4: 5/5 Boltz repeats
  9WHV: 5/5 Boltz repeats
  9E74: 5/5 Boltz repeats
  9E75: 5/5 Boltz repeats
  9G4J: 5/5 Boltz repeats
  9KGG: 5/5 Boltz repeats
  9EBP: 5/5 Boltz repeats
  9LJN: 5/5 Boltz repeats
  9ZCC: 0/5 Boltz repeats

ASSEMBLING FINAL HYBRID SUBMISSION
  0/28  8ZNQ  | 0.0s
  10/28  9G4R  | 0.1s
  20/28  9WHV  | 0.2s

✓ submission.csv saved

===== S

/bin/sh: 1: USalign: not found
